# Proyecto final · Notebook 1 · Extracción de muestras

## Clasificación de cobertura y uso del suelo — Distrito de Villa Hayes

**Qué hace este notebook:** arma la tabla de entrenamiento. No entrena nada todavía.
Al final va a quedar un CSV donde cada fila es un píxel con su clase y unas 50 columnas
de features espectrales. Ese CSV es la entrada del Notebook 2.

---

### Las tres decisiones de diseño, y por qué

**1. Las etiquetas son anuales, las features son multitemporales.**

Una etiqueta es una propiedad del lugar, no de la imagen: si un polígono es bosque, es
bosque en marzo y en agosto. Lo que cambia entre fechas es la *firma espectral*, no la
clase. Por eso no necesitamos una etiqueta por imagen — necesitamos una etiqueta por
sitio y por año, y muchas observaciones espectrales de ese sitio.

Además, clasificar fecha por fecha sería *peor*: con una sola imagen, un lote de soja en
pleno desarrollo y una pastura bien verde son casi idénticos. Lo que los separa es el
ciclo anual (la agrícola va suelo desnudo → verde intenso → cosecha; la pastura es
estable). Es el mismo argumento de estacionalidad del Módulo 6, aplicado al píxel en vez
de a la serie.

Solución: **dos compuestos por año** (temporada seca y temporada húmeda). El modelo ve
las dos caras del mismo píxel y puede aprender esa dinámica.

**2. Las etiquetas salen de MapBiomas, la validación va a salir de vos.**

MapBiomas Chaco tiene ruido, pero Random Forest tolera bien el ruido en el
entrenamiento: con miles de muestras, el error se promedia. Lo que *no* se puede hacer es
validar contra MapBiomas, porque ahí sólo mediríamos cuánto nos parecemos a MapBiomas.
La validación honesta viene de un set independiente fotointerpretado a mano (lo armás vos
en paralelo, ver la celda final).

**3. El split va a ser espacial, no aleatorio.**

Este es el equivalente espacial de "el futuro no entrena al pasado". Píxeles vecinos son
casi idénticos: si caen algunos en train y otros en test, el accuracy sale inflado (95%
ficticio). Por eso acá le asignamos a cada muestra un **ID de bloque espacial**, y el
Notebook 2 va a partir por bloques enteros, no por píxeles sueltos.

---

### Fuentes de datos

| Qué | Fuente | Resolución | Período |
|---|---|---|---|
| Imágenes | Sentinel-2 SR Harmonized | 10–20 m | 2019 en adelante |
| Etiquetas | MapBiomas Chaco Colección 5 | 30 m | 1985–2023 |
| Límite distrital | FAO GAUL nivel 2 | vectorial | — |

**Años elegidos: 2019, 2021, 2023.** MapBiomas Chaco llega hasta 2023, así que esos son
los años con etiqueta disponible. 2025 queda para el Notebook 3 como *predicción* sobre
un año que el producto de referencia no cubre.

---
## 0 · Instalación y autenticación

`earthengine-api` es el cliente de Python. `geemap` agrega los mapas interactivos y
algunas utilidades para pasar de Earth Engine a pandas.

En Colab estas dos suelen estar preinstaladas o se instalan en unos segundos.

In [ ]:
# Si ya las tenés instaladas, esta celda no hace nada.
!pip install -q earthengine-api geemap

In [ ]:
import ee
import geemap
import pandas as pd
import numpy as np

# ⚠️ CAMBIAR: introducir el ID de tu proyecto de Google Cloud habilitado para Earth Engine.
PROYECTO_GEE = "tu-proyecto-gee"

# Authenticate() abre una ventana para dar permiso. Solo hace falta la primera vez
# por sesión de Colab; si ya estás autenticado, tira un aviso y sigue.
try:
    ee.Initialize(project=PROYECTO_GEE)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROYECTO_GEE)

print("Earth Engine inicializado.")
print("Prueba de conexión:", ee.Number(1).add(1).getInfo(), "(debería decir 2)")

---
## 1 · Parámetros del proyecto

Todo lo que se toca está acá arriba. Si más adelante querés cambiar años, temporadas o
cantidad de muestras, se cambia en esta celda y se vuelve a correr todo.

**Sobre las temporadas:** estamos en el hemisferio sur y en el Chaco Bajo. La época de
lluvias va de noviembre a marzo y la seca de junio a septiembre. Para el año `Y`, la
temporada húmeda arranca en diciembre del año anterior — por eso la ventana cruza el año
calendario.

In [ ]:
# --- Años a procesar (los tres tienen etiqueta de MapBiomas Chaco) ---
ANIOS = [2019, 2021, 2023]

# --- Ventanas de temporada (mes-día). La húmeda arranca en dic del año anterior. ---
TEMPORADAS = {
    "humeda": {"inicio": "12-01", "fin": "03-31", "cruza_anio": True},
    "seca":   {"inicio": "06-01", "fin": "09-30", "cruza_anio": False},
}

# --- Bandas de Sentinel-2 que vamos a usar ---
# Dejamos afuera B1 (aerosol), B9 (vapor de agua) y B10 (cirrus): son bandas de
# corrección atmosférica, no aportan información de cobertura.
BANDAS_S2 = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]

# --- Muestreo ---
MUESTRAS_POR_CLASE = 500   # por clase y por año
ESCALA_MUESTREO = 10       # metros, nativo de Sentinel-2
SEMILLA = 42

# --- Bloques espaciales para el split del Notebook 2 ---
# 0.05° ≈ 5.5 km. Los bloques tienen que ser bastante más grandes que la escala a la
# que los píxeles están correlacionados entre sí, o el split no sirve de nada.
TAMANIO_BLOQUE_GRADOS = 0.05

# --- Bloques reservados para validación ---
# 1 de cada 5 bloques queda AFUERA del entrenamiento. Los puntos de validación que vas a
# verificar a mano salen solo de esos bloques: así la métrica final es sobre territorio
# que el modelo nunca vio, ni siquiera de cerca.
RESERVA_CADA = 5
CANDIDATOS_VALIDACION_POR_CLASE = 20   # verificás estos y descartás los dudosos
ANIO_VALIDACION = 2023

# --- Assets ---
ASSET_MAPBIOMAS = (
    "projects/mapbiomas-public/assets/chaco/lulc/collection5/"
    "mapbiomas_chaco_collection5_integration_v2"
)
COLECCION_S2 = "COPERNICUS/S2_SR_HARMONIZED"

print(f"Años: {ANIOS}")
print(f"Temporadas: {list(TEMPORADAS.keys())}")
print(f"Bandas: {len(BANDAS_S2)} → {BANDAS_S2}")

---
## 2 · Área de estudio

**Cambio respecto a la primera versión:** FAO GAUL 2015 tiene la división distrital
vieja de Presidente Hayes (solo 4 distritos). Su "Villa Hayes" incluye territorio que hoy
pertenece a otros distritos creados después, y por eso salía un área enorme.

Usamos **geoBoundaries**, que trae los límites distritales actuales. Primero listamos los
distritos de la zona para confirmar el nombre exacto.

In [ ]:
adm2 = ee.FeatureCollection("WM/geoLab/geoBoundaries/600/ADM2")

# Todos los distritos de Paraguay; mostramos los que coinciden con la zona del Bajo Chaco
distritos_py = adm2.filter(ee.Filter.eq("shapeGroup", "PRY"))
nombres = distritos_py.aggregate_array("shapeName").getInfo()

claves = ["Hayes", "Aceval", "Nanawa", "Falc", "Bruguez", "Pinasco", "Pozo"]
print("Distritos del Bajo Chaco en geoBoundaries:")
for n in sorted(nombres):
    if any(k.lower() in n.lower() for k in claves):
        print("  -", n)

In [ ]:
# ⚠️ Si el nombre que imprimió la celda anterior es distinto, cambialo acá.
NOMBRE_DISTRITO = "Villa Hayes"

aoi_fc = distritos_py.filter(ee.Filter.eq("shapeName", NOMBRE_DISTRITO))
aoi = aoi_fc.geometry()

n_encontrados = aoi_fc.size().getInfo()
if n_encontrados == 0:
    raise ValueError(
        f"No se encontró el distrito '{NOMBRE_DISTRITO}'. "
        f"Elegí uno de la lista que imprimió la celda anterior."
    )

area_km2 = aoi.area(maxError=100).divide(1e6).getInfo()
print(f"Distrito: {NOMBRE_DISTRITO}")
print(f"Superficie: {area_km2:,.0f} km²")
print("\n(Con GAUL 2015 salían ~35 000 km². Si acá también sale un número de ese orden,")
print(" el límite sigue siendo el viejo: avisame antes de seguir.)")

---
## 3 · Enmascarado de nubes y compuestos por temporada

### Por qué hace falta enmascarar

Una nube tiene una firma espectral muy parecida a la de un suelo desnudo muy reflectivo, y
una sombra de nube se parece al agua. Si no las sacamos, el modelo va a aprender a
clasificar nubes, no coberturas.

Sentinel-2 L2A trae la banda **SCL** (Scene Classification Layer), que ya viene con una
clasificación gruesa por píxel hecha por el procesador de ESA. Nos alcanza para descartar
lo que no sirve:

| Código SCL | Qué es | ¿Lo usamos? |
|---|---|---|
| 0, 1 | sin datos, saturado | ✗ |
| 3 | sombra de nube | ✗ |
| 8, 9 | nube media y alta probabilidad | ✗ |
| 10 | cirrus | ✗ |
| 11 | nieve/hielo | ✗ |
| 2, 4, 5, 6, 7 | oscuro, vegetación, suelo, agua, sin clasificar | ✓ |

### Por qué mediana y no media

El compuesto se arma con la **mediana** de todas las imágenes válidas de la ventana. La
mediana es robusta: si quedó alguna nube sin enmascarar, es un valor extremo y la mediana
lo ignora. La media, en cambio, se dejaría arrastrar por ese valor.

In [ ]:
def enmascarar_nubes_s2(img):
    """Descarta píxeles de nube, sombra y cirrus usando la banda SCL, y escala la
    reflectancia al rango 0-1.

    Sentinel-2 SR guarda la reflectancia multiplicada por 10000 para ahorrar espacio
    (usa enteros en vez de decimales). Dividimos para volver a valores físicos.
    """
    scl = img.select("SCL")

    # remap(): a los códigos que queremos conservar les asignamos 1, al resto 0.
    mascara_valida = (
        scl.remap([2, 4, 5, 6, 7], [1, 1, 1, 1, 1], 0).eq(1)
    )

    return (
        img.select(BANDAS_S2)
        .divide(10000)
        .updateMask(mascara_valida)
        .copyProperties(img, ["system:time_start"])
    )


def compuesto_temporada(anio, nombre_temporada):
    """Devuelve la mediana de Sentinel-2 para una temporada de un año, con las bandas
    renombradas con el sufijo de la temporada (ej: B4_seca).

    El sufijo importa: cuando juntemos las dos temporadas en una sola imagen, necesitamos
    poder distinguir el B4 de la seca del B4 de la húmeda.
    """
    cfg = TEMPORADAS[nombre_temporada]

    # La temporada húmeda arranca en diciembre del año anterior.
    anio_inicio = anio - 1 if cfg["cruza_anio"] else anio
    fecha_inicio = f"{anio_inicio}-{cfg['inicio']}"
    fecha_fin = f"{anio}-{cfg['fin']}"

    coleccion = (
        ee.ImageCollection(COLECCION_S2)
        .filterBounds(aoi)
        .filterDate(fecha_inicio, fecha_fin)
        # Descartamos de entrada las escenas con más de 60% de nubes: no aportan
        # casi nada y hacen más lento el cálculo.
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
        .map(enmascarar_nubes_s2)
    )

    nuevos_nombres = [f"{b}_{nombre_temporada}" for b in BANDAS_S2]

    return (
        coleccion.median()
        .rename(nuevos_nombres)
        .set("n_imagenes", coleccion.size())
        .set("temporada", nombre_temporada)
        .set("anio", anio)
    )

In [ ]:
# Chequeo de disponibilidad: cuántas imágenes hay en cada ventana.
# Si alguna temporada tiene muy pocas imágenes, el compuesto va a tener huecos y
# conviene ampliar la ventana antes de seguir.
print("Imágenes disponibles por año y temporada:\n")
for anio in ANIOS:
    for temp in TEMPORADAS:
        n = compuesto_temporada(anio, temp).get("n_imagenes").getInfo()
        estado = "ok" if n >= 10 else "⚠️ pocas"
        print(f"  {anio} · {temp:7s}: {n:3d} imágenes   {estado}")

---
## 4 · Índices espectrales

Las bandas crudas ya traen información, pero los índices la concentran. Un índice es un
cociente normalizado entre dos bandas, diseñado para que una cobertura en particular se
destaque.

Esto es **feature engineering** en el sentido exacto de la Tarea 5 del curso: no agregamos
datos nuevos, recombinamos los que ya tenemos de una forma que al modelo le resulta más
fácil de usar.

| Índice | Fórmula | Para qué sirve |
|---|---|---|
| **NDVI** | (NIR − Rojo)/(NIR + Rojo) | vegetación verde y vigorosa |
| **NDWI** | (Verde − NIR)/(Verde + NIR) | agua abierta |
| **MNDWI** | (Verde − SWIR1)/(Verde + SWIR1) | agua, pero sin confundirse con construido |
| **NDBI** | (SWIR1 − NIR)/(SWIR1 + NIR) | superficies construidas |
| **BSI** | ver código | suelo desnudo |
| **SAVI** | NDVI ajustado por suelo | vegetación rala (clave en Chaco) |

**Por qué MNDWI además de NDWI:** el NDWI clásico confunde agua con superficies urbanas.
El MNDWI usa SWIR en vez de NIR y separa bien las dos. Como tenemos ambas clases en el
área, conviene tener los dos índices y dejar que el modelo decida cuál usa — eso lo vamos
a ver después con SHAP.

**Por qué SAVI:** en el Chaco hay mucha vegetación rala donde el suelo se ve entre las
plantas. El NDVI se "contamina" con la reflectancia del suelo; el SAVI mete un factor de
corrección (L = 0.5) para compensarlo.

In [ ]:
def agregar_indices(img, temporada):
    """Calcula los índices espectrales sobre un compuesto de temporada y los agrega
    como bandas nuevas."""
    s = temporada  # alias corto para los nombres de banda

    azul  = img.select(f"B2_{s}")
    verde = img.select(f"B3_{s}")
    rojo  = img.select(f"B4_{s}")
    nir   = img.select(f"B8_{s}")
    swir1 = img.select(f"B11_{s}")
    swir2 = img.select(f"B12_{s}")

    # normalizedDifference() hace (a - b)/(a + b), que es la forma de casi todos
    # los índices. Le pasamos las bandas en orden.
    ndvi  = nir.subtract(rojo).divide(nir.add(rojo)).rename(f"NDVI_{s}")
    ndwi  = verde.subtract(nir).divide(verde.add(nir)).rename(f"NDWI_{s}")
    mndwi = verde.subtract(swir1).divide(verde.add(swir1)).rename(f"MNDWI_{s}")
    ndbi  = swir1.subtract(nir).divide(swir1.add(nir)).rename(f"NDBI_{s}")

    # BSI (Bare Soil Index): combina cuatro bandas. Sube donde hay suelo expuesto.
    bsi = (
        swir1.add(rojo).subtract(nir.add(azul))
        .divide(swir1.add(rojo).add(nir).add(azul))
        .rename(f"BSI_{s}")
    )

    # SAVI: NDVI corregido por el brillo del suelo. L=0.5 es el valor estándar
    # para vegetación de densidad intermedia.
    L = 0.5
    savi = (
        nir.subtract(rojo).multiply(1 + L)
        .divide(nir.add(rojo).add(L))
        .rename(f"SAVI_{s}")
    )

    return img.addBands([ndvi, ndwi, mndwi, ndbi, bsi, savi])

---
## 5 · Features de diferencia entre temporadas

Acá está la parte que hace que este trabajo sea multitemporal de verdad.

Un cultivo anual y una pastura pueden tener NDVI casi idéntico en la temporada húmeda.
Lo que los separa es **cuánto cambian** entre la húmeda y la seca: el cultivo se cosecha y
su NDVI se desploma, la pastura baja mucho menos.

Esa diferencia es una feature que ninguna imagen individual contiene. Es exactamente la
lógica del Ejercicio 3 de la Tarea 5: construir una columna nueva que capture una
dinámica, en vez de esperar que el modelo la deduzca solo.

In [ ]:
def agregar_diferencias_estacionales(img):
    """Agrega la diferencia húmeda − seca para los índices que capturan dinámica
    de vegetación y humedad."""
    diferencias = []
    for indice in ["NDVI", "NDWI", "MNDWI", "BSI", "SAVI"]:
        d = (
            img.select(f"{indice}_humeda")
            .subtract(img.select(f"{indice}_seca"))
            .rename(f"d{indice}")
        )
        diferencias.append(d)
    return img.addBands(diferencias)


def construir_stack(anio):
    """Arma la imagen completa de features para un año: las dos temporadas con sus
    bandas e índices, más las diferencias estacionales."""
    humeda = agregar_indices(compuesto_temporada(anio, "humeda"), "humeda")
    seca   = agregar_indices(compuesto_temporada(anio, "seca"), "seca")

    stack = humeda.addBands(seca)
    stack = agregar_diferencias_estacionales(stack)

    return stack.clip(aoi).set("anio", anio)


# Verificación: cuántas features quedaron y cómo se llaman.
stack_prueba = construir_stack(ANIOS[0])
nombres_features = stack_prueba.bandNames().getInfo()

print(f"Total de features: {len(nombres_features)}\n")
print("Temporada húmeda:", [b for b in nombres_features if b.endswith("_humeda")])
print()
print("Temporada seca:  ", [b for b in nombres_features if b.endswith("_seca")])
print()
print("Diferencias:     ", [b for b in nombres_features if b.startswith("d")])

---
## 6 · Etiquetas de MapBiomas

MapBiomas Chaco tiene 18 clases. Nosotros queremos 7. Antes de reclasificar conviene ver
**qué códigos aparecen realmente en Villa Hayes** — no tiene sentido armar una clase para
algo que en el distrito no existe.

La celda siguiente cuenta los píxeles de cada código presente en el área.

In [ ]:
mapbiomas = ee.Image(ASSET_MAPBIOMAS)

# Cada año es una banda, con el nombre 'classification_AAAA'.
bandas_mb = mapbiomas.bandNames().getInfo()
print(f"MapBiomas Chaco C5 · {len(bandas_mb)} bandas anuales")
print(f"Primera: {bandas_mb[0]}  ·  Última: {bandas_mb[-1]}\n")

# Histograma de códigos presentes en el distrito para el primer año
banda_ref = f"classification_{ANIOS[0]}"

histograma = (
    mapbiomas.select(banda_ref)
    .clip(aoi)
    .reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=aoi,
        scale=30,
        maxPixels=1e10,
        bestEffort=True,
    )
    .get(banda_ref)
    .getInfo()
)

total_px = sum(histograma.values())
print(f"Códigos de MapBiomas presentes en {NOMBRE_DISTRITO} ({ANIOS[0]}):\n")
print(f"{'Código':>8} {'Píxeles':>12} {'% del área':>12}")
print("-" * 34)
for codigo, n in sorted(histograma.items(), key=lambda x: -x[1]):
    print(f"{codigo:>8} {int(n):>12,} {100*n/total_px:>11.2f}%")

### Reclasificación a nuestras 7 clases

Los códigos de MapBiomas siguen un esquema común a todas las iniciativas de la red.
Estos son los relevantes para el Chaco:

| Código MapBiomas Chaco C5 | Descripción | → Nuestra clase |
|---|---|---|
| 3, 4, 6, 45 | Bosque cerrado, abierto, inundable, disperso | **1 · Bosque** |
| 9 | Plantación forestal | **1 · Bosque** |
| 11 | Pastizal inundable | **2 · Humedal** |
| 12, 42, 43, 44 | Pastizal (cerrado, abierto, ralo) | **3 · Pastura** |
| 15 | Pastura implantada | **3 · Pastura** |
| 57, 58, 36 | Cultivo simple, múltiple, plantación arbustiva | **4 · Agrícola** |
| 22, 23, 25, 61 | Sin vegetación, arena, salitral | **5 · Suelo desnudo** |
| 24 | Área urbana | **6 · Urbano** |
| 26 | Cuerpos de agua | **7 · Agua** |
| 14, 27 | Agropecuario sin distinguir, no observado | descartado |

Fuente: tabla de leyenda del ATBD de la Colección 5 de MapBiomas Chaco. Ojo que los
códigos **no** coinciden del todo con MapBiomas Brasil: el agua es 26, no 33.

**⚠️ Verificá contra el histograma de arriba.** Si aparece algún código con peso
importante que no está en esta tabla, hay que agregarlo — la lista de MapBiomas Chaco
puede tener alguna clase propia de la región. Los códigos que no estén mapeados quedan
como 0 y se descartan.

**Sobre la clase Humedal:** la agregamos porque el Bajo Chaco tiene mucho pastizal
inundable, y sin esta clase esos píxeles se repartirían entre pastura y agua ensuciando
las dos. Es la clase donde esperamos más confusión — lo vamos a ver en la matriz del
Notebook 2.

In [ ]:
# Diccionario de reclasificación: código MapBiomas → nuestra clase
RECLASIFICACION = {
    # Bosque: cerrado (3), abierto (4), inundable (6), disperso (45), plantación (9)
    3: 1, 4: 1, 6: 1, 45: 1, 9: 1,
    # Humedal: pastizal inundable (11)
    11: 2,
    # Pastura y pastizal: pastizal (12), cerrado (43), abierto (42), ralo (44), pastura (15)
    12: 3, 42: 3, 43: 3, 44: 3, 15: 3,
    # Agrícola: cultivo simple (57), múltiple (58), plantación arbustiva (36)
    57: 4, 58: 4, 36: 4,
    # Suelo desnudo: sin vegetación (22), playa/arena (23), otros (25), salitral (61)
    22: 5, 23: 5, 25: 5, 61: 5,
    # Urbano
    24: 6,
    # Agua (en MapBiomas Chaco es 26, no 33 como en Brasil)
    26: 7,
    # Quedan afuera a propósito (→ 0): 14 "agropecuario sin distinguir", que mezcla
    # pastura y cultivo y ensuciaría las dos clases; y 27 "no observado".
}

NOMBRES_CLASES = {
    1: "Bosque",
    2: "Humedal",
    3: "Pastura",
    4: "Agricola",
    5: "Suelo desnudo",
    6: "Urbano",
    7: "Agua",
}

PALETA_CLASES = {
    1: "#1f8d49",  # verde oscuro
    2: "#519799",  # verde azulado
    3: "#d6bc74",  # ocre
    4: "#e974ed",  # magenta
    5: "#d4271e",  # rojo
    6: "#54231c",  # marrón oscuro
    7: "#2532e4",  # azul
}

codigos_origen = list(RECLASIFICACION.keys())
codigos_destino = list(RECLASIFICACION.values())


def etiquetas_anio(anio):
    """Devuelve la imagen de etiquetas reclasificadas para un año."""
    return (
        mapbiomas.select(f"classification_{anio}")
        .remap(codigos_origen, codigos_destino, 0)  # lo no mapeado → 0
        .rename("clase")
        .clip(aoi)
    )


# Verificación: distribución de nuestras clases
hist_clases = (
    etiquetas_anio(ANIOS[0])
    .reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=aoi, scale=30, maxPixels=1e10, bestEffort=True,
    )
    .get("clase").getInfo()
)

total = sum(hist_clases.values())
print(f"Distribución de clases en {NOMBRE_DISTRITO} ({ANIOS[0]}):\n")
print(f"{'Clase':<16} {'Píxeles':>12} {'%':>8}")
print("-" * 38)
for cod, n in sorted(hist_clases.items(), key=lambda x: -x[1]):
    nombre = NOMBRES_CLASES.get(int(cod), "(sin mapear)")
    print(f"{nombre:<16} {int(n):>12,} {100*n/total:>7.2f}%")

print("\n⚠️ Si '(sin mapear)' tiene más del 5%, revisá el diccionario RECLASIFICACION.")

---
## 7 · Filtro de pureza

Acá resolvemos un problema concreto: **MapBiomas está a 30 m y Sentinel-2 a 10 m.**

Un píxel de MapBiomas de 30 m que cae justo en el borde entre un monte y un cultivo tiene
una sola etiqueta, pero adentro contiene las dos coberturas. Si muestreamos ahí, le damos
al modelo una firma espectral mezclada con una etiqueta pura: ruido puro.

**Solución:** solo muestreamos donde la vecindad de 3×3 píxeles de MapBiomas (90 × 90 m)
tiene una única clase. Eso garantiza que estamos bien adentro de una cobertura homogénea
y no sobre un borde.

El costo es que perdemos las zonas de transición. Lo aceptamos: es mejor entrenar con
menos muestras pero limpias, que con más muestras ambiguas.

In [ ]:
def etiquetas_puras(anio):
    """Etiquetas del año, enmascaradas para dejar solo píxeles en zonas homogéneas."""
    etiquetas = etiquetas_anio(anio)

    # countDistinctNonNull sobre una ventana de 3x3: cuántos valores distintos hay.
    # Si da 1, toda la vecindad es de la misma clase.
    # Ojo: sin el reproject, Earth Engine calcularía la vecindad en la escala a la que
    # se pide el resultado (10 m al muestrear), y la ventana 3x3 cubriría solo 30 m.
    # Forzamos la proyección nativa de MapBiomas para que la ventana sea de 90 x 90 m.
    n_distintos = etiquetas.reduceNeighborhood(
        reducer=ee.Reducer.countDistinctNonNull(),
        kernel=ee.Kernel.square(radius=1, units="pixels"),
    ).reproject(mapbiomas.projection())

    es_homogeneo = n_distintos.eq(1)
    tiene_clase = etiquetas.gt(0)  # descartamos el 0 (sin mapear)

    return etiquetas.updateMask(es_homogeneo.And(tiene_clase))


# Cuánto sobrevive al filtro
hist_puro = (
    etiquetas_puras(ANIOS[0])
    .reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=aoi, scale=30, maxPixels=1e10, bestEffort=True,
    )
    .get("clase").getInfo()
)

print(f"Píxeles disponibles después del filtro de pureza ({ANIOS[0]}):\n")
print(f"{'Clase':<16} {'Antes':>12} {'Después':>12} {'Sobrevive':>11}")
print("-" * 54)
for cod in sorted(NOMBRES_CLASES):
    antes = int(hist_clases.get(str(cod), 0))
    despues = int(hist_puro.get(str(cod), 0))
    pct = 100 * despues / antes if antes > 0 else 0
    alerta = "  ⚠️" if despues < 500 else ""
    print(f"{NOMBRES_CLASES[cod]:<16} {antes:>12,} {despues:>12,} {pct:>10.1f}%{alerta}")

print("\nLas clases con ⚠️ tienen pocos píxeles: puede que no alcancen para el muestreo.")
print("Si pasa, bajá MUESTRAS_POR_CLASE o fusioná esa clase con otra.")

---
## 8 · Bloques espaciales

Este es el paso que hace honesta la validación del Notebook 2.

Cada muestra recibe el ID de la celda de una grilla regular de ~5.5 km. Después, al
partir train/test, vamos a repartir **bloques enteros**, nunca píxeles sueltos. Así
garantizamos que un píxel de test nunca tenga un vecino inmediato en train.

Sin esto, dos píxeles contiguos (prácticamente el mismo lugar) pueden caer uno en cada
lado, y el modelo "acierta" porque ya vio ese sitio. Es la misma fuga que el Ejercicio 5
de la Tarea 4, donde la feature `gasto_por_cliente_hoy` subía el R² a 0.91 y era mentira.

In [ ]:
def banda_bloque():
    """Imagen de una banda con el ID del bloque espacial de cada píxel.

    Tomamos la longitud y latitud de cada píxel, las dividimos por el tamaño de bloque y
    truncamos: todos los píxeles de la misma celda quedan con el mismo par de índices.
    Después combinamos los dos índices en un solo número entero.
    """
    lonlat = ee.Image.pixelLonLat()

    # +10000 para que los índices sean positivos (estamos en lon y lat negativas)
    ix = lonlat.select("longitude").divide(TAMANIO_BLOQUE_GRADOS).floor().add(10000)
    iy = lonlat.select("latitude").divide(TAMANIO_BLOQUE_GRADOS).floor().add(10000)

    return ix.multiply(100000).add(iy).toInt64().rename("bloque")


def banda_reserva():
    """1 en los bloques reservados para validación, 0 en los de entrenamiento.

    Mezclamos los índices de columna y fila (ix*7 + iy*3) para que los bloques reservados
    queden salteados como un tablero, y no en franjas enteras de la misma latitud.
    """
    lonlat = ee.Image.pixelLonLat()
    ix = lonlat.select("longitude").divide(TAMANIO_BLOQUE_GRADOS).floor().add(10000)
    iy = lonlat.select("latitude").divide(TAMANIO_BLOQUE_GRADOS).floor().add(10000)
    return ix.multiply(7).add(iy.multiply(3)).mod(RESERVA_CADA).eq(0).rename("reserva")


# Cuántos bloques cubren el distrito
n_bloques = (
    banda_bloque().clip(aoi)
    .reduceRegion(
        reducer=ee.Reducer.countDistinctNonNull(),
        geometry=aoi, scale=100, maxPixels=1e10, bestEffort=True,
    )
    .get("bloque").getInfo()
)

print(f"El distrito se cubre con ~{n_bloques} bloques de {TAMANIO_BLOQUE_GRADOS}° "
      f"(≈{TAMANIO_BLOQUE_GRADOS * 111:.1f} km de lado).")
print("\nCon esa cantidad se puede hacer un GroupKFold de 5 folds sin problemas.")
print("Si fueran menos de ~20, habría que achicar TAMANIO_BLOQUE_GRADOS.")

---
## 9 · Muestreo estratificado

`stratifiedSample` toma la misma cantidad de píxeles por clase, en vez de muestrear al
azar sobre todo el territorio.

**Por qué estratificado y no aleatorio:** si muestreáramos al azar, las clases
dominantes (probablemente bosque y pastura) se llevarían casi todas las muestras, y el
urbano — que ocupa una fracción mínima del distrito — quedaría con un puñado de píxeles.
El modelo aprendería a ignorarlo. Es el mismo problema de desbalanceo del Escenario C de
la Tarea M3.

Cada muestra sale con: las ~50 features, su clase, su bloque, el año y sus coordenadas.

In [ ]:
def muestrear_anio(anio):
    """Devuelve una FeatureCollection de muestras etiquetadas para un año.

    Solo muestrea en bloques de ENTRENAMIENTO: los reservados quedan enmascarados.
    """
    stack = construir_stack(anio)
    etiquetas = etiquetas_puras(anio).updateMask(banda_reserva().Not())
    bloques = banda_bloque()

    # Juntamos todo en una sola imagen: features + etiqueta + bloque.
    # stratifiedSample va a devolver todas las bandas como propiedades.
    imagen_completa = stack.addBands(etiquetas).addBands(bloques)

    muestras = imagen_completa.stratifiedSample(
        numPoints=MUESTRAS_POR_CLASE,
        classBand="clase",
        region=aoi,
        scale=ESCALA_MUESTREO,
        seed=SEMILLA + anio,   # semilla distinta por año
        geometries=True,       # queremos las coordenadas
        tileScale=8,           # parte el cálculo en más pedazos: evita quedarse sin memoria
        dropNulls=True,        # descarta píxeles con huecos en el compuesto
    )

    # Guardamos año y coordenadas como columnas comunes (más fácil de leer en pandas)
    def con_coords(f):
        c = f.geometry().coordinates()
        return f.set({"anio": anio, "lon": c.get(0), "lat": c.get(1)})

    return muestras.map(con_coords)


# Acá NO pedimos .size().getInfo(): eso obligaría a Earth Engine a calcular todo el
# muestreo en modo interactivo, que tiene un límite de tiempo y se corta. El cálculo
# real lo hace la exportación de la sección siguiente, que no tiene ese límite.
muestras_por_anio = {anio: muestrear_anio(anio) for anio in ANIOS}
print("Muestreo definido para:", list(muestras_por_anio))


---
## 10 · Exportación a Drive

Con 3 años × 7 clases × 500 muestras salen ~10 000 filas. Earth Engine no deja bajar más
de 5 000 elementos de forma interactiva, así que **exportamos a Drive**: es una tarea en
segundo plano, sin límite de tiempo.

Lanzamos **una tarea por año** (corren en paralelo y si una falla no arrastra a las
otras) y **una más con los candidatos de validación**.

### Los candidatos de validación

En vez de que dibujes 300 puntos desde cero, el notebook te propone 20 puntos por clase
tomados de los **bloques reservados** (los que el modelo nunca ve). Vos solo los
verificás sobre imagen de alta resolución y completás la columna `clase_verificada`.
Verificar es mucho más rápido que digitalizar.

Importante: la columna `clase_mapbiomas` es solo una pista para que sepas qué mirar.
**Tu respuesta tiene que salir de la imagen, no de esa columna.** Si un punto es dudoso,
dejá `clase_verificada` vacía y se descarta.

In [ ]:
CARPETA_DRIVE = "proyecto_final_ml"
tareas = []

# 1) Una tarea por año con las muestras de entrenamiento
for anio, fc in muestras_por_anio.items():
    t = ee.batch.Export.table.toDrive(
        collection=fc,
        description=f"muestras_{anio}",
        folder=CARPETA_DRIVE,
        fileNamePrefix=f"muestras_{anio}",
        fileFormat="CSV",
        selectors=nombres_features + ["clase", "bloque", "anio", "lon", "lat"],
    )
    t.start()
    tareas.append(t)

# 2) Candidatos de validación: solo bloques reservados, año de validación
etiq_val = etiquetas_puras(ANIO_VALIDACION).updateMask(banda_reserva())
candidatos = etiq_val.rename("clase_mapbiomas").stratifiedSample(
    numPoints=CANDIDATOS_VALIDACION_POR_CLASE,
    classBand="clase_mapbiomas",
    region=aoi,
    scale=30,
    seed=SEMILLA,
    geometries=True,
    tileScale=8,
)

def preparar_candidato(f):
    c = f.geometry().coordinates()
    return f.set({"lon": c.get(0), "lat": c.get(1), "clase_verificada": ""})

t = ee.batch.Export.table.toDrive(
    collection=candidatos.map(preparar_candidato),
    description="candidatos_validacion",
    folder=CARPETA_DRIVE,
    fileNamePrefix="candidatos_validacion",
    fileFormat="CSV",
    selectors=["lon", "lat", "clase_mapbiomas", "clase_verificada"],
)
t.start()
tareas.append(t)

print(f"{len(tareas)} tareas lanzadas. Seguí el progreso en:")
print("  https://code.earthengine.google.com/tasks")
print("\nSuelen tardar entre 10 y 40 minutos. Podés correr la celda de abajo para ver el estado.")


In [ ]:
# Estado de las tareas. Volvé a correr esta celda cada tanto hasta que todas digan COMPLETED.
for t in tareas:
    st = t.status()
    linea = f"  {st['description']:<24} {st['state']}"
    if st["state"] == "FAILED":
        linea += f"  ← {st.get('error_message', '')}"
    print(linea)


In [ ]:
# Cuando todas las tareas estén COMPLETED: montamos Drive y juntamos los tres años.
from google.colab import drive
drive.mount("/content/drive")

ruta = f"/content/drive/MyDrive/{CARPETA_DRIVE}"
df = pd.concat(
    [pd.read_csv(f"{ruta}/muestras_{anio}.csv") for anio in ANIOS],
    ignore_index=True,
)
print(f"✓ {len(df):,} filas × {len(df.columns)} columnas")
print(df["anio"].value_counts().sort_index().to_string())


---
## 11 · Control de calidad de la tabla

Antes de pasar al modelo, tres chequeos. Si alguno falla, es mejor volver atrás ahora que
descubrirlo con el modelo ya entrenado.

In [ ]:
if df is not None:
    # Limpieza: geemap deja una columna con la geometría serializada que no necesitamos
    # como feature, pero sí queremos las coordenadas.
    if ".geo" in df.columns:
        import json
        coords = df[".geo"].apply(lambda g: json.loads(g)["coordinates"])
        df["lon"] = coords.apply(lambda c: c[0])
        df["lat"] = coords.apply(lambda c: c[1])
        df = df.drop(columns=[".geo"])
    if "system:index" in df.columns:
        df = df.drop(columns=["system:index"])

    print("=" * 60)
    print("1. BALANCE DE CLASES")
    print("=" * 60)
    conteo = df["clase"].value_counts().sort_index()
    for cod, n in conteo.items():
        print(f"  {NOMBRES_CLASES.get(cod, cod):<16} {n:>6,}")
    desbalance = conteo.max() / conteo.min()
    print(f"\n  Razón max/min: {desbalance:.1f}x", end="  ")
    print("(ok)" if desbalance < 3 else "⚠️ revisar: alguna clase quedó corta")

    print("\n" + "=" * 60)
    print("2. VALORES FALTANTES")
    print("=" * 60)
    faltantes = df.isna().sum()
    faltantes = faltantes[faltantes > 0]
    if len(faltantes) == 0:
        print("  Sin faltantes.")
    else:
        print(faltantes.to_string())
        print("\n  ⚠️ Vienen de huecos en el compuesto (nubes persistentes).")

    print("\n" + "=" * 60)
    print("3. BLOQUES ESPACIALES")
    print("=" * 60)
    n_bloq = df["bloque"].nunique()
    print(f"  Bloques distintos con muestras: {n_bloq}")
    print(f"  Muestras por bloque (mediana): {df['bloque'].value_counts().median():.0f}")
    print("\n  " + ("ok: alcanza para GroupKFold de 5 folds" if n_bloq >= 20
                   else "⚠️ pocos bloques: achicá TAMANIO_BLOQUE_GRADOS"))

In [ ]:
if df is not None:
    # Separabilidad: ¿las clases se distinguen en las features clave?
    # Si dos clases tienen medias casi iguales en todos los índices, el modelo
    # tampoco las va a poder separar.
    indices_clave = ["NDVI_humeda", "NDVI_seca", "dNDVI",
                     "MNDWI_humeda", "NDBI_seca", "BSI_seca"]
    disponibles = [c for c in indices_clave if c in df.columns]

    resumen = df.groupby("clase")[disponibles].mean().round(3)
    resumen.index = [NOMBRES_CLASES.get(i, i) for i in resumen.index]

    print("Media de los índices clave por clase:\n")
    print(resumen.to_string())
    print("\nQué mirar:")
    print("  · Agua debería tener MNDWI_humeda bien alto y NDVI bajo.")
    print("  · Agrícola debería tener el dNDVI más alto (cambia mucho entre temporadas).")
    print("  · Pastura debería tener dNDVI bajo (es estable todo el año).")
    print("  · Urbano y Suelo desnudo van a parecerse: es la confusión esperable.")

In [ ]:
if df is not None:
    df.to_csv("muestras_villa_hayes.csv", index=False)
    # Copia en Drive: la sesión de Colab se borra, Drive no.
    df.to_csv(f"/content/drive/MyDrive/{CARPETA_DRIVE}/muestras_villa_hayes.csv", index=False)
    print(f"✓ Guardado: muestras_villa_hayes.csv  ({len(df):,} filas)")
    print(f"  También en Drive: {CARPETA_DRIVE}/muestras_villa_hayes.csv")


---
## 12 · Mapa de control

Última verificación, visual: ¿los compuestos se ven bien? ¿las etiquetas caen donde
corresponde?

Mirá especialmente que no haya franjas o huecos raros en el compuesto — serían zonas
donde no quedó ninguna imagen sin nubes.

In [ ]:
anio_vis = ANIOS[-1]
stack_vis = construir_stack(anio_vis)

Mapa = geemap.Map()
Mapa.centerObject(aoi, 9)

# Color verdadero, temporada seca
Mapa.addLayer(
    stack_vis, {"bands": ["B4_seca", "B3_seca", "B2_seca"], "min": 0.02, "max": 0.25},
    f"Color verdadero · seca {anio_vis}",
)

# Falso color infrarrojo: la vegetación sana sale roja. Útil para ver a ojo
# dónde hay monte y dónde cultivo.
Mapa.addLayer(
    stack_vis, {"bands": ["B8_humeda", "B4_humeda", "B3_humeda"], "min": 0.02, "max": 0.4},
    f"Falso color · húmeda {anio_vis}", False,
)

# dNDVI: dónde cambia más la vegetación entre temporadas (agrícola en rojo)
Mapa.addLayer(
    stack_vis.select("dNDVI"),
    {"min": -0.3, "max": 0.3, "palette": ["blue", "white", "red"]},
    "dNDVI (húmeda − seca)", False,
)

# Etiquetas de MapBiomas reclasificadas
Mapa.addLayer(
    etiquetas_anio(anio_vis),
    {"min": 1, "max": 7, "palette": [PALETA_CLASES[i] for i in sorted(PALETA_CLASES)]},
    f"Etiquetas MapBiomas {anio_vis}",
)

# Solo los píxeles que sobrevivieron al filtro de pureza
Mapa.addLayer(
    etiquetas_puras(anio_vis),
    {"min": 1, "max": 7, "palette": [PALETA_CLASES[i] for i in sorted(PALETA_CLASES)]},
    "Etiquetas puras (zonas homogéneas)", False,
)

Mapa.addLayer(aoi_fc.style(color="yellow", fillColor="00000000", width=2), {}, "Distrito")

Mapa

---
## 13 · Lo que sigue

### Archivos que quedaron en tu Drive (carpeta `proyecto_final_ml`)
- `muestras_2019.csv`, `muestras_2021.csv`, `muestras_2023.csv` → la tabla de entrenamiento
  (también guardada junta como `muestras_villa_hayes.csv` en la sección 11).
- `candidatos_validacion.csv` → los puntos para verificar a mano.

### Verificación de los candidatos (~40 min)

1. En QGIS: *Capa → Añadir capa → Texto delimitado*, X = `lon`, Y = `lat`, EPSG:4326.
2. Fondo: Google Satellite o Bing (plugin QuickMapServices). Si tenés dudas sobre el año,
   compará con el compuesto 2023 en Earth Engine.
3. Abrí la tabla de atributos, recorré punto por punto y completá `clase_verificada`
   con el código 1–7. Si dudás, dejala vacía.
4. Guardá como CSV con el nombre `validacion_verificada.csv`.

Códigos: 1 Bosque · 2 Humedal · 3 Pastura · 4 Agrícola · 5 Suelo desnudo · 6 Urbano · 7 Agua

**Regla:** tu respuesta sale de la imagen, no de `clase_mapbiomas`. Si copiás la columna,
la validación vuelve a ser circular y pierde todo su valor.

### Después
`muestras_villa_hayes.csv` y `validacion_verificada.csv` alimentan `notebook_final.ipynb`,
que corre sin Earth Engine: Pipeline + ColumnTransformer, Random Forest contra XGBoost con
`GroupKFold` por bloques, SHAP, y la métrica final contra tus puntos verificados.